# 📖 Notebook 3: Consistent Hashing for Shards

In Notebook 1 we saw the **resharding problem**: with simple `hash % N`, adding one shard moves ~75% of the data. That's catastrophic in production.

**Consistent hashing** solves this. When you add or remove a shard, only **~1/N** of the data needs to move (where N is the number of shards). This makes scaling much safer.

## How It Works

Imagine a circle (a "hash ring") numbered 0 to 360. Each shard gets placed at a position on the ring by hashing its name. Each data key is also hashed to a position, and it belongs to the **next shard clockwise** on the ring.

## Learning Objectives

By the end of this notebook, you'll understand:
- How a consistent hash ring works
- How to implement one from scratch in Python
- How adding/removing shards minimizes data movement
- Why virtual nodes improve balance

## 🛠️ Setup

Make sure infrastructure is running:

```bash
cd 01-foundations/sharding
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import hashlib
import bisect
import psycopg2
import random

SHARD_CONFIGS = {
    0: {"host": "localhost", "port": 55433, "database": "shard_1", "user": "demo", "password": "demo"},
    1: {"host": "localhost", "port": 55434, "database": "shard_2", "user": "demo", "password": "demo"},
    2: {"host": "localhost", "port": 55435, "database": "shard_3", "user": "demo", "password": "demo"},
}

def get_connection(shard_id):
    return psycopg2.connect(**SHARD_CONFIGS[shard_id])

for sid in SHARD_CONFIGS:
    conn = get_connection(sid)
    cur = conn.cursor()
    cur.execute("SELECT current_database()")
    print(f"✅ Shard {sid} → {cur.fetchone()[0]}")
    cur.close()
    conn.close()

## Step 1: Build a Consistent Hash Ring

The key data structure is a **sorted list of positions on the ring**. Each shard gets one or more positions. To find which shard owns a key, we hash the key and find the next position clockwise.

In [ ]:
class ConsistentHashRing:
    """
    A consistent hash ring for routing keys to shards.
    
    Each shard can have multiple 'virtual nodes' on the ring
    to improve distribution evenness.
    """
    
    def __init__(self, num_virtual_nodes=1):
        self.num_virtual_nodes = num_virtual_nodes
        self.ring = {}          # position → shard_id
        self.sorted_keys = []   # sorted list of positions on the ring
        self.shards = set()     # set of shard IDs on the ring
    
    def _hash(self, key):
        """Hash a key to a position on the ring (0 to 2^32-1)."""
        return int(hashlib.md5(str(key).encode()).hexdigest(), 16) % (2**32)
    
    def add_shard(self, shard_id):
        """Add a shard to the ring with its virtual nodes."""
        self.shards.add(shard_id)
        for i in range(self.num_virtual_nodes):
            # Each virtual node gets a unique key like 'shard_0_vn_0'
            virtual_key = f"shard_{shard_id}_vn_{i}"
            position = self._hash(virtual_key)
            self.ring[position] = shard_id
            bisect.insort(self.sorted_keys, position)
    
    def remove_shard(self, shard_id):
        """Remove a shard and all its virtual nodes from the ring."""
        self.shards.discard(shard_id)
        positions_to_remove = [pos for pos, sid in self.ring.items() if sid == shard_id]
        for pos in positions_to_remove:
            del self.ring[pos]
            self.sorted_keys.remove(pos)
    
    def get_shard(self, key):
        """
        Find which shard a key belongs to.
        Hash the key, then walk clockwise to the next shard position.
        """
        if not self.ring:
            raise Exception("No shards on the ring!")
        
        position = self._hash(key)
        # Find the first shard position >= our key's position
        idx = bisect.bisect_right(self.sorted_keys, position)
        # Wrap around if we're past the last position (it's a ring!)
        if idx >= len(self.sorted_keys):
            idx = 0
        return self.ring[self.sorted_keys[idx]]

# Create a ring with 3 shards, 1 virtual node each (for now)
ring = ConsistentHashRing(num_virtual_nodes=1)
ring.add_shard(0)
ring.add_shard(1)
ring.add_shard(2)

print("Hash Ring positions:")
print("─" * 50)
for pos in ring.sorted_keys:
    shard = ring.ring[pos]
    # Show position as % of ring
    pct = pos / (2**32) * 100
    print(f"  Position {pos:>12d} ({pct:5.1f}% of ring) → Shard {shard}")

## Step 2: Test Key Distribution

Let's distribute 1,000 keys and see how they land.

In [ ]:
def check_distribution(ring, num_keys=1000):
    """Count how many keys end up on each shard."""
    counts = {sid: 0 for sid in ring.shards}
    for i in range(num_keys):
        shard = ring.get_shard(f"user_{i}")
        counts[shard] += 1
    return counts

counts = check_distribution(ring)

print("Distribution with 1 virtual node per shard:")
print("─" * 45)
for sid, count in sorted(counts.items()):
    pct = count / 1000 * 100
    bar = '█' * int(pct / 2)
    print(f"  Shard {sid}: {count:4d} keys ({pct:5.1f}%) {bar}")

print()
print("⚠️  With only 1 virtual node, distribution can be very uneven!")
print("   We'll fix this with more virtual nodes in Step 4.")

## Step 3: Adding a Shard — Minimal Data Movement 🎉

This is the magic of consistent hashing. Let's add a 4th shard and see how many keys move.

In [ ]:
# Record where each key lives BEFORE adding shard 3
before = {}
for i in range(1000):
    key = f"user_{i}"
    before[key] = ring.get_shard(key)

# Add a 4th shard
ring.add_shard(3)
print(f"Added Shard 3. Ring now has {len(ring.shards)} shards.")
print()

# Check where each key lives AFTER
after = {}
moved = 0
for i in range(1000):
    key = f"user_{i}"
    after[key] = ring.get_shard(key)
    if before[key] != after[key]:
        moved += 1

print(f"Keys that stayed on the same shard: {1000 - moved} ({(1000-moved)/10:.0f}%)")
print(f"Keys that moved to a new shard:     {moved} ({moved/10:.0f}%)")
print()

# Compare with simple modulo
modulo_moved = 0
for i in range(1000):
    key_bytes = str(f"user_{i}").encode()
    h = int(hashlib.md5(key_bytes).hexdigest()[:8], 16)
    old_shard = h % 3
    new_shard = h % 4
    if old_shard != new_shard:
        modulo_moved += 1

print("Comparison:")
print("─" * 50)
print(f"  Consistent hashing: {moved/10:.0f}% of keys moved")
print(f"  Simple hash % N:    {modulo_moved/10:.0f}% of keys moved")
print()
print("✅ Consistent hashing moves MUCH less data when adding shards!")

## Step 4: Virtual Nodes for Better Balance

With only 1 virtual node per shard, the distribution is uneven because each shard only occupies one point on the ring. **Virtual nodes** place each shard at multiple positions, spreading the load more evenly.

In [ ]:
print("Impact of virtual nodes on distribution (1000 keys, 3 shards):")
print("═" * 60)

for vn_count in [1, 10, 50, 150]:
    test_ring = ConsistentHashRing(num_virtual_nodes=vn_count)
    test_ring.add_shard(0)
    test_ring.add_shard(1)
    test_ring.add_shard(2)
    
    counts = check_distribution(test_ring)
    values = list(counts.values())
    min_v, max_v = min(values), max(values)
    spread = max_v - min_v
    
    print(f"\n  {vn_count:3d} virtual nodes per shard:")
    for sid in sorted(counts):
        c = counts[sid]
        bar = '█' * (c // 20)
        print(f"    Shard {sid}: {c:4d} {bar}")
    print(f"    Spread: {spread} (lower is better)")

print()
print("More virtual nodes = more even distribution.")
print("150 virtual nodes per shard is typical in production (e.g., Cassandra).")

## Step 5: Use It with Real Databases

Let's put it all together — use the consistent hash ring to route writes and reads to our actual Postgres shards.

In [ ]:
# Create a production-style ring with 150 virtual nodes
prod_ring = ConsistentHashRing(num_virtual_nodes=150)
prod_ring.add_shard(0)
prod_ring.add_shard(1)
prod_ring.add_shard(2)

# Clear all shards
for sid in SHARD_CONFIGS:
    conn = get_connection(sid)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute("DELETE FROM orders")
    cur.execute("DELETE FROM users")
    cur.close()
    conn.close()

COUNTRIES = ['US', 'UK', 'Germany', 'Japan', 'Brazil', 'India', 'Canada', 'France']

# Insert 300 users using consistent hashing
shard_counts = {i: 0 for i in range(3)}

for user_id in range(1, 301):
    shard_id = prod_ring.get_shard(user_id)
    shard_counts[shard_id] += 1
    
    conn = get_connection(shard_id)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO users (id, username, email, country) VALUES (%s, %s, %s, %s)",
        (user_id, f"user_{user_id}", f"user{user_id}@example.com", random.choice(COUNTRIES))
    )
    cur.close()
    conn.close()

print("Consistent hashing distribution (150 virtual nodes):")
print("─" * 50)
for sid, count in shard_counts.items():
    bar = '█' * (count // 3)
    print(f"  Shard {sid}: {count:3d} users {bar}")
print(f"  Total: {sum(shard_counts.values())} users")

# Verify a lookup
test_id = 42
shard_id = prod_ring.get_shard(test_id)
conn = get_connection(shard_id)
cur = conn.cursor()
cur.execute("SELECT id, username, email FROM users WHERE id = %s", (test_id,))
result = cur.fetchone()
cur.close()
conn.close()
print(f"\n🔍 Lookup user {test_id} → Shard {shard_id} → {result}")

## Step 6: Removing a Shard

Consistent hashing also helps when removing shards. Only the keys on the removed shard need to move to their new "next clockwise" shard.

In [ ]:
# Check assignments before removing shard 2
before = {}
for i in range(1, 301):
    before[i] = prod_ring.get_shard(i)

# Remove shard 2
prod_ring.remove_shard(2)
print("Removed Shard 2 from the ring.")
print()

# Check who moved
moved_count = 0
moved_from_2 = 0
for i in range(1, 301):
    new_shard = prod_ring.get_shard(i)
    if before[i] != new_shard:
        moved_count += 1
        if before[i] == 2:
            moved_from_2 += 1

new_counts = {0: 0, 1: 0}
for i in range(1, 301):
    new_counts[prod_ring.get_shard(i)] += 1

print(f"Total keys that moved: {moved_count}")
print(f"Keys from removed Shard 2: {moved_from_2}")
print(f"Keys from Shard 0 or 1 that moved: {moved_count - moved_from_2}")
print()
print("New distribution across remaining shards:")
for sid, count in new_counts.items():
    bar = '█' * (count // 3)
    print(f"  Shard {sid}: {count:3d} users {bar}")
print()
print("Only keys from the removed shard needed to move.")
print("Keys on Shard 0 and Shard 1 stayed put.")

# Restore shard 2 for subsequent notebooks
prod_ring.add_shard(2)

## 🎯 Key Takeaways

1. **Consistent hashing** arranges shards on a ring; keys belong to the next shard clockwise
2. **Adding a shard** moves only ~1/N of the data (vs ~75% with simple modulo)
3. **Removing a shard** only moves the data from that shard — other shards are unaffected
4. **Virtual nodes** (100–200 per shard) are essential for even distribution
5. Production systems like **Cassandra** use consistent hashing with virtual nodes

### Next Up

**Notebook 4: Rebalancing Strategies** — How to actually move data between shards when you scale.